## analyze_chain_results.ipynb

**Purpose:** Merge device execution logs from the CasFly IoT chain experiments and generate quantitative analysis and figures for the paper.

Specifically produces figures and data for:
- **Fig 3**: Variation of total time with chain length and number of paths (`variation_metrics.csv`)
- **Fig 4**: Device performance across devices — time per device, resource consumption by disease
- **Fig 6**: Storage and load-time comparison of CPT vs TPHG structures

**Inputs required:**
- `device_partitions_patientwise1_rasp/` — per-device execution logs (`*_metrics_log.csv`) from Raspberry Pi runs
- `raspresults1/` — supplemental metrics logs
- `results.csv` — merged/pre-built results table (produced by the merge step below)
- `device_metrics_analysiscptloadtime.csv` — CPT load time measurements
- `greedy_walk_summary_with_resources.csv` — baseline comparison data

**Outputs:**
- `newallmerged_metrics_log.csv` — merged metrics from all devices
- `variation_metrics.csv` — chain length vs time summary table
- `device_disease_metrics.csv`, `device_metrics_overall.csv` — per-device/disease stats
- `resource_metrics_by_device_*.csv` — CPU/memory resource usage tables
- `device_usage_metrics.csv`, `device_usage_global.csv` — device participation metrics
- Paper figures (PNG/PDF)

## Section 1: Discover and Merge Metrics Logs

Scan `device_partitions_patientwise1_rasp/` and `raspresults1/` for all `*_metrics_log.csv` files.
Concatenate them into `newallmerged_metrics_log.csv`, tagged by device type.

In [ ]:
import pandas as pd
import glob

metrics_files = glob.glob(f"device_partitions_patientwise1_rasp/**/*_metrics_log.csv", recursive=True)
print(metrics_files)

if metrics_files:
    print(f"Found {len(metrics_files)} files.")
else:
    print("No matching files found.")


import pandas as pd
import glob

metrics_filesrasp = glob.glob(f"raspresults1/*_metrics_log.csv", recursive=True)
print(metrics_filesrasp)

if metrics_filesrasp:
    print(f"Found {len(metrics_filesrasp)} files.")
else:
    print("No matching files found.")


In [ ]:

dataframes = []


for file in metrics_files:
    
    df = pd.read_csv(file)
    
    df['device_type'] = 'mac'
    dataframes.append(df)


for file in metrics_filesrasp:
    
    df = pd.read_csv(file)
    
    df['device_type'] = 'rasp'
    dataframes.append(df)


merged_df = pd.concat(dataframes, ignore_index=True)


if 'Timestamp' in merged_df.columns:
    merged_df['Timestamp'] = pd.to_datetime(merged_df['Timestamp'])
    merged_df.sort_values(by='Timestamp', inplace=True)
    merged_df.reset_index(drop=True, inplace=True)


merged_df.to_csv("newallmerged_metrics_log.csv", index=False)
print(merged_df.head())

## Section 2: Load results.csv and Compute Chain/Path Features

Load the merged `results.csv`. Extract the number of paths from Chain Data and chain length from No. of Devices Accessed. Generate a 3-D scatter to visualize the relationship between chain length, path count, and total execution time (Fig 3 overview).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import t
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


df = pd.read_csv("results.csv")


df = df[df['Total Time (T)'] > 0]


def extract_number_of_paths(chain_data):
    try:
        chain = eval(chain_data)
        return len(chain)
    except:
        return 0

df['Number of Paths'] = df['Chain Data'].apply(extract_number_of_paths)


df['Chain Length'] = df['No. of Devices Accessed'].apply(lambda x: len(eval(x)) if pd.notnull(x) else 0)
print(df['Chain Length'])

x = df['Chain Length']
y = df['Number of Paths']
z = df['Total Time (T)']


fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')


scatter = ax.scatter(x, y, z, c=z, cmap='viridis', s=50, edgecolor='k', alpha=0.7)


ax.set_xlabel('Chain Length', fontsize=12)
ax.set_ylabel('Number of Paths', fontsize=12)
ax.set_zlabel('Total Time (T)', fontsize=12)
ax.set_title('3D Scatter Plot: Chain Length vs Number of Paths vs Total Time', fontsize=14)


cbar = fig.colorbar(scatter, ax=ax, pad=0.1)
cbar.set_label('Total Time (T)', fontsize=12)


plt.tight_layout()
plt.savefig("3dplot.png")
plt.show()




## Section 3: Variation Metrics — Chain Length vs Total Time (Fig 3)

Compute quantitative summary statistics for "Variation of total time with chain length and number of paths."

Outputs: `variation_metrics.csv` and `variation_metrics.tex`.

In [ ]:
import ast
import re
import numpy as np

# --- helper: safe parse for list-like strings ---
def _lit(s):
    try:
        return ast.literal_eval(str(s).replace("inf","1e9"))
    except Exception:
        return []

# 1) Ensure Total Time exists and is positive
time_col = None
for c in df.columns:
    if re.search(r"total\s*time", c, flags=re.I):
        time_col = c; break
if time_col is None:
    raise KeyError("Could not find a 'Total Time' column. Available: " + ", ".join(df.columns))

df[time_col] = pd.to_numeric(df[time_col], errors="coerce")
df = df[df[time_col] > 0].copy()

# 2) Ensure Chain Data exists
chain_col = None
for c in df.columns:
    if re.search(r"chain\s*data", c, flags=re.I):
        chain_col = c; break
if chain_col is None:
    raise KeyError("Could not find 'Chain Data'. Available: " + ", ".join(df.columns))

# 3) Derive Number of Paths if missing
if "Number of Paths" not in df.columns:
    df["Number of Paths"] = df[chain_col].apply(lambda s: len(_lit(s)))

# 4) Derive Chain Length (unique devices) if missing
def chain_len_from_chaindata(s):
    lst = _lit(s)
    # device name is last element in each entry (as in your plot code)
    devs = {e[-1] for e in lst if isinstance(e, (list, tuple)) and len(e) > 4}
    return len(devs)

if "Chain Length" not in df.columns:
    if "No. of Devices Accessed" in df.columns:
        # sometimes this column holds a list of devices as a string
        df["Chain Length"] = df["No. of Devices Accessed"].apply(
            lambda x: len(_lit(x)) if pd.notnull(x) else np.nan
        )
        # fill any NaNs from Chain Data
        df["Chain Length"] = df["Chain Length"].fillna(df[chain_col].apply(chain_len_from_chaindata))
    else:
        df["Chain Length"] = df[chain_col].apply(chain_len_from_chaindata)

# 5) Normalize column names so the rest of your script works verbatim
df.rename(columns={time_col: "Total Time (T)", chain_col: "Chain Data"}, inplace=True)

# 6) Keep rows with all fields
df = df.dropna(subset=["Chain Length", "Number of Paths", "Total Time (T)"]).reset_index(drop=True)


In [ ]:
#!/usr/bin/env python3
# Quantitative metrics for "Variation of total time with chain length and number of paths"
# Inputs: results.csv  (expects columns: "Chain Data", "Total Time (T)", "Number of Paths")
# Outputs:
#   - variation_metrics.tex  (LaTeX macros)
#   - variation_metrics.csv  (human-readable summary table)

import ast
import re
import numpy as np

# --- helper: safe parse for list-like strings ---
def _lit(s):
    try:
        return ast.literal_eval(str(s).replace("inf","1e9"))
    except Exception:
        return []

# 1) Ensure Total Time exists and is positive
time_col = None
for c in df.columns:
    if re.search(r"total\s*time", c, flags=re.I):
        time_col = c; break
if time_col is None:
    raise KeyError("Could not find a 'Total Time' column. Available: " + ", ".join(df.columns))

df[time_col] = pd.to_numeric(df[time_col], errors="coerce")
df = df[df[time_col] > 0].copy()

# 2) Ensure Chain Data exists
chain_col = None
for c in df.columns:
    if re.search(r"chain\s*data", c, flags=re.I):
        chain_col = c; break
if chain_col is None:
    raise KeyError("Could not find 'Chain Data'. Available: " + ", ".join(df.columns))

# 3) Derive Number of Paths if missing
if "Number of Paths" not in df.columns:
    df["Number of Paths"] = df[chain_col].apply(lambda s: len(_lit(s)))

# 4) Derive Chain Length (unique devices) if missing
def chain_len_from_chaindata(s):
    lst = _lit(s)
    # device name is last element in each entry (as in your plot code)
    devs = {e[-1] for e in lst if isinstance(e, (list, tuple)) and len(e) > 4}
    return len(devs)

if "Chain Length" not in df.columns:
    if "No. of Devices Accessed" in df.columns:
        # sometimes this column holds a list of devices as a string
        df["Chain Length"] = df["No. of Devices Accessed"].apply(
            lambda x: len(_lit(x)) if pd.notnull(x) else np.nan
        )
        # fill any NaNs from Chain Data
        df["Chain Length"] = df["Chain Length"].fillna(df[chain_col].apply(chain_len_from_chaindata))
    else:
        df["Chain Length"] = df[chain_col].apply(chain_len_from_chaindata)

# 5) Normalize column names so the rest of your script works verbatim
df.rename(columns={time_col: "Total Time (T)", chain_col: "Chain Data"}, inplace=True)

# 6) Keep rows with all fields
df = df.dropna(subset=["Chain Length", "Number of Paths", "Total Time (T)"]).reset_index(drop=True)

# bins for legend buckets
def bucket(paths):
    p = int(paths)
    if p <= 5: return "1–5"
    if p <= 15: return "6–15"
    return "16+"
df["PathsBin"] = df["Number of Paths"].apply(bucket)

# correlations
pear_len = stats.pearsonr(df["Chain Length"], df["Total Time (T)"])
spear_len = stats.spearmanr(df["Chain Length"], df["Total Time (T)"])
pear_paths = stats.pearsonr(df["Number of Paths"], df["Total Time (T)"])
spear_paths = stats.spearmanr(df["Number of Paths"], df["Total Time (T)"])

# regression: log10(total time) ~ chain length + log10(paths+1)
X = np.column_stack([df["Chain Length"].values, np.log10(df["Number of Paths"].values + 1.0)])
y = np.log10(df["Total Time (T)"].values)
reg = LinearRegression().fit(X, y)
r2 = reg.score(X, y)
beta_len, beta_paths = reg.coef_.tolist()
intercept = reg.intercept_

# robust per-bin summaries
summary_rows = []
for b in ["1–5","6–15","16+"]:
    sub = df[df["PathsBin"] == b]["Total Time (T)"].values
    if sub.size:
        median = np.median(sub)
        q1, q3 = np.percentile(sub, [25, 75])
        iqr = q3 - q1
        p90 = np.percentile(sub, 90)
        n = sub.size
    else:
        median = iqr = p90 = n = np.nan
    summary_rows.append({"Bin": b, "N": n, "Median": median, "IQR": iqr, "P90": p90})

summary = pd.DataFrame(summary_rows)

# outliers: above 95th percentile overall
p95 = np.percentile(df["Total Time (T)"], 95)
outlier_frac = float((df["Total Time (T)"] > p95).mean())

# write CSV
summary_out = pd.DataFrame({
    "Pearson(chain_len, time)": [pear_len.statistic],
    "Spearman(chain_len, time)": [spear_len.correlation],
    "Pearson(num_paths, time)": [pear_paths.statistic],
    "Spearman(num_paths, time)": [spear_paths.correlation],
    "Reg: beta_len (log-time)": [beta_len],
    "Reg: beta_paths (log-time)": [beta_paths],
    "Reg: R^2": [r2],
    "OutlierRate(>95th)": [outlier_frac],
    "N": [len(df)]
})
summary_out.to_csv("variation_metrics.csv", index=False)

# # write LaTeX macros
# def cmd(name, val, fmt="{:.3f}"):
#     try:
#         if val is None or (isinstance(val, float) and (np.isnan(val) or np.isinf(val))):
#             return f"\\newcommand\\{name}{{NA}}\n"
#         return f"\\newcommand\\{name}{{{fmt.format(val)}}}\n"
#     except Exception:
#         return f"\\newcommand\\{name}{{NA}}\n"

# def get_bin(b, col, fmt="{:.3f}"):
#     v = summary.loc[summary["Bin"]==b, col]
#     return (fmt.format(float(v.iloc[0])) if len(v) else "NA")

# with open("variation_metrics.tex","w") as f:
#     f.write(cmd("VarN", len(df), "{:d}"))
#     f.write(cmd("PearLen", pear_len.statistic))
#     f.write(cmd("SpearLen", spear_len.correlation))
#     f.write(cmd("PearPaths", pear_paths.statistic))
#     f.write(cmd("SpearPaths", spear_paths.correlation))
#     f.write(cmd("BetaLen", beta_len))
#     f.write(cmd("BetaPaths", beta_paths))
#     f.write(cmd("VarRTwo", r2))
#     f.write(cmd("OutlierFracPct", 100*outlier_frac, "{:.1f}"))
#     # bin summaries
#     for b, tag in [("1–5","B1"), ("6–15","B2"), ("16+","B3")]:
#         f.write(f"\\newcommand\\{tag}N{{{get_bin(b,'N','{:d}')}}}\n")
#         f.write(f"\\newcommand\\{tag}Med{{{get_bin(b,'Median')}}}\n")
#         f.write(f"\\newcommand\\{tag}IQR{{{get_bin(b,'IQR')}}}\n")
#         f.write(f"\\newcommand\\{tag}Pninety{{{get_bin(b,'P90')}}}\n")

print("Wrote variation_metrics.tex and variation_metrics.csv")


## Section 4: Timing Breakdown by Disease (Fig 4 — Grouped Bar)

Group execution timing metrics (TPHG load, Viterbi, fallback) by disease name. Compute 95% CI error bars and generate grouped bar charts for the paper.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats


df = pd.read_csv("results.csv")


metrics = ["Cached TPHG Load Time (t1)", "Backward Viterbi Time (t2)", "Fallback Path Time (t_fallback)"]


grouped_df = df.groupby("Disease Name")[metrics].agg(['mean', 'count', 'std']).reset_index()


for metric in metrics:
    grouped_df[f"{metric} 95% CI"] = stats.t.ppf(0.975, grouped_df[(metric, 'count')] - 1) * grouped_df[(metric, 'std')] / np.sqrt(grouped_df[(metric, 'count')])


grouped_df.columns = [' '.join(col).strip() if col[1] else col[0] for col in grouped_df.columns]


font0 = {'family': 'Times New Roman', 'size': 16}
font1 = {'family': 'Times New Roman', 'size': 20}
font2 = {'family': 'Times New Roman', 'size': 18}

x = np.arange(len(grouped_df["Disease Name"]))  
bar_width = 0.25  
patterns = ["///", "\\\\", "---"]  


fig, ax = plt.subplots(figsize=(12, 7))


for i, metric in enumerate(metrics):
    ax.bar(
        x + i * bar_width, 
        grouped_df[f"{metric} mean"], 
        width=bar_width, 
        yerr=grouped_df[f"{metric} 95% CI"], 
        label=metric, 
        hatch=patterns[i], 
        capsize=5
    )


ax.set_xticks(x + bar_width)
ax.set_xticklabels(grouped_df["Disease Name"], rotation=45, ha='right', **font2)
ax.set_xlabel("Disease Name", **font1)
ax.set_ylabel("Time (s)", **font1)

ax.legend(
    fontsize=12, 
    loc="upper center", 
    bbox_to_anchor=(0.65, 1.00), 
    ncol=1, 
)
ax.grid(axis="y", linestyle="--", alpha=0.7)


plt.tight_layout()


plt.savefig("diseasebars_updated.png")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


font0 = {'family': 'Times New Roman', 'size': 28}
font1 = {'family': 'Times New Roman', 'size': 32}
font2 = {'family': 'Times New Roman', 'size': 30}
font3 = {'family': 'Times New Roman', 'size': 33}


df = pd.read_csv("results.csv")
wrapped_labels = [
    "\n".join(textwrap.wrap(disease_name, width=15)) for disease_name in merged_df["Disease Name"]
]

disease_name_mapping = {
    "Myocardialinfarction(disorder)": "Myocardial\nInfarction",
    "Non-smallcelllungcancer(disorder)": "Non-small\nCell Lung\nCancer",
    "Seizuredisorder": "Seizure\nDisorder",
    "Stroke": "Stroke",
    "Suspectedlungcancer(situation)": "Suspected\nLung Cancer"
}
df['Disease Name'] = df['Disease Name'].replace(disease_name_mapping)


if 'current_device' in df.columns:
    df['device_number'] = df['current_device'].str.extract(r'(\d+)')
    df['device_number'] = pd.to_numeric(df['device_number'], errors='coerce')
    df = df.dropna(subset=['device_number'])
    df['device_number'] = df['device_number'].astype(int)
    df = df.sort_values(by='device_number').drop(columns=['device_number'])
else:
    print("Warning: 'current_device' column not found in the dataset. Sorting skipped.")


df["Chain Data"] = df["Chain Data"].str.replace("inf", "1e9")


filtered_df = df[df["Total Time (T)"] > 0]


def extract_device_names(chain_data):
    try:
        
        chain_list = eval(chain_data)
        
        return [entry[4] for entry in chain_list if len(entry) > 4]
    except Exception as e:
        print(f"Error parsing chain data: {chain_data}")
        print(f"Error details: {e}")
        return []


filtered_df["Device Names"] = filtered_df["Chain Data"].apply(extract_device_names)


device_counts_by_disease = {}


for _, row in filtered_df.iterrows():
    disease = row["Disease Name"]
    devices = row["Device Names"]
    if disease not in device_counts_by_disease:
        device_counts_by_disease[disease] = {}
    for device in devices:
        if device not in device_counts_by_disease[disease]:
            device_counts_by_disease[disease][device] = 0
        device_counts_by_disease[disease][device] += 1


data = []
for disease, devices in device_counts_by_disease.items():
    for device, count in devices.items():
        data.append([disease, device, count])

plot_df = pd.DataFrame(data, columns=["Disease Name", "Device Name", "Count"])


diseases = plot_df["Disease Name"].unique()
devices = sorted(plot_df["Device Name"].unique(), key=lambda x: int(x[6:]))
patterns = ['//', "*", 'O' "", 'o', '|', '...', '--', '**'] * (len(devices) // 9 + 1)
colors = plt.cm.tab10(np.linspace(0, 1, len(devices)))
bar_width = 0.8 / len(devices)  


plt.figure(figsize=(14, 10))
x_positions = np.arange(len(diseases))

for i, device in enumerate(devices):
    device_data = plot_df[plot_df["Device Name"] == device]
    counts = [device_data[device_data["Disease Name"] == disease]["Count"].sum() for disease in diseases]
    plt.bar(
        x_positions + i * bar_width, 
        counts, 
        width=bar_width, 
        label=device, 
        color=colors[i % len(colors)], 
        hatch=patterns[i % len(patterns)], 
        edgecolor="black"
    )


plt.xlabel("Disease Name", fontsize=font0['size'], family=font0['family'])
plt.ylabel("Count", fontsize=font0['size'], family=font0['family'])
plt.xticks(x_positions + bar_width * (len(devices) - 1) / 2, diseases, rotation=0, ha='center', fontsize=font2['size'], family=font2['family'])
plt.yticks(fontsize=font2['size'], family=font2['family'])
plt.legend(fontsize=font0['size'], title_fontsize=font3['size'], loc="upper center", ncol=2)
plt.grid(axis="y", linestyle="--", alpha=0.7)


plt.tight_layout()
plt.savefig("separate_device_usage_bars.png")
plt.show()


## Section 5: Device Performance Across Devices (Fig 4 — Boxplot)

Visualize `Time per Device (t_dash)` distributions across all devices and diseases. Exports:
- `device_disease_metrics.csv` — per-device per-disease stats
- `device_metrics_overall.csv` — overall device statistics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch


file_path = "results.csv"  
df = pd.read_csv(file_path)


columns_to_keep = ['current_device', 'Disease Name', 'Time per Device (t_dash)']
df = df[columns_to_keep]


df['Time per Device (t_dash)'] = pd.to_numeric(df['Time per Device (t_dash)'], errors='coerce')


df = df.dropna(subset=['Time per Device (t_dash)'])
df = df[df['Time per Device (t_dash)'] != 0]


df = df[df['Time per Device (t_dash)'] <= 10]


disease_name_mapping = {
    "Myocardialinfarction(disorder)": "Myocardial Infarction",
    "Non-smallcelllungcancer(disorder)": "Non-small Cell Lung Cancer",
    "Seizuredisorder": "Seizure Disorder",
    "Stroke": "Stroke",
    "Suspectedlungcancer(situation)": "Suspected Lung Cancer"
}
df['Disease Name'] = df['Disease Name'].replace(disease_name_mapping)


df['current_device'] = df['current_device'].replace(
    {"Device3": "Device3\n(IoT device)", "Device26": "Device26\n(IoT device)"}
)


df['device_number'] = df['current_device'].str.extract(r'(\d+)').astype(int)
df = df.sort_values(by='device_number').drop(columns=['device_number'])


disease_styles = {
    "Myocardial Infarction": {'color': 'lightblue', 'hatch': '/'},
    "Non-small Cell Lung Cancer": {'color': 'lightgreen', 'hatch': '*'},
    "Seizure Disorder": {'color': 'lightpink', 'hatch': '|'},
    "Stroke": {'color': 'lightyellow', 'hatch': '-'},
    "Suspected Lung Cancer": {'color': 'lightcoral', 'hatch': '+'},
}

font0 = {'family': 'Times New Roman', 'size': 32}
font1 = {'family': 'Times New Roman', 'size': 36}
font2 = {'family': 'Times New Roman', 'size': 34}
font3 = {'family': 'Times New Roman', 'size': 37}


devices = df['current_device'].unique()
diseases = df['Disease Name'].unique()
box_data = {disease: [df[(df['current_device'] == device) & (df['Disease Name'] == disease)]['Time per Device (t_dash)'].values
                      for device in devices] for disease in diseases}


plt.figure(figsize=(20, 15))
x_positions = np.arange(len(devices))  
box_width = 0.15  
offsets = np.arange(len(diseases)) * box_width - (len(diseases) - 1) * box_width / 2  


for i, disease in enumerate(diseases):
    bplot = plt.boxplot(
        box_data[disease],
        positions=x_positions + offsets[i],
        widths=box_width,
        patch_artist=True,
        showfliers=False,  
        boxprops=dict(facecolor=disease_styles[disease]['color'], hatch=disease_styles[disease]['hatch']),
        medianprops=dict(color='black'),
    )


legend_handles = [
    Patch(facecolor=style['color'], edgecolor='black', hatch=style['hatch'], label=disease)
    for disease, style in disease_styles.items()
]
plt.legend(handles=legend_handles, loc="upper right", fontsize=font3['size'])


plt.xticks(x_positions, devices, rotation=45, fontsize=font2['size'], ha='center')
plt.yticks(fontsize=font2['size'])
plt.xlabel("Device Name", **font1)
plt.ylabel("Total Time per Device (s)", **font1)

plt.grid(axis='y', linestyle='--', alpha=0.7)


plt.tight_layout()
plt.savefig("time_per_device_boxplot_with_fonts.png", bbox_inches="tight")
plt.show()


In [ ]:
#!/usr/bin/env python3
# Quantitative stats for "Device performance across devices" boxplot
# Inputs:  results.csv  (expects: current_device, Disease Name, Time per Device (t_dash))
# Outputs: device_disease_metrics.csv, device_metrics_overall.csv, disease_metrics_overall.csv, device_perf_highlights.txt

import pandas as pd
import numpy as np
import re

IN_CSV = "results.csv"

# --------------------------
# 1) Load & clean
# --------------------------
keep_cols = ["current_device", "Disease Name", "Time per Device (t_dash)"]
df = pd.read_csv(IN_CSV)

missing = [c for c in keep_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns in results.csv: {missing}. Found columns: {list(df.columns)}")

df = df[keep_cols].copy()
df["Time per Device (t_dash)"] = pd.to_numeric(df["Time per Device (t_dash)"], errors="coerce")

# drop NaN/zero, cap at 10s like your plotting code
df = df.dropna(subset=["Time per Device (t_dash)"])
df = df[df["Time per Device (t_dash)"] > 0]
df = df[df["Time per Device (t_dash)"] <= 10].copy()

# Disease name normalization (edit/extend if needed)
name_map = {
    "Myocardialinfarction(disorder)": "Myocardial Infarction",
    "Non-smallcelllungcancer(disorder)": "Non-small Cell Lung Cancer",
    "Seizuredisorder": "Seizure Disorder",
    "Stroke": "Stroke",
    "Suspectedlungcancer(situation)": "Suspected Lung Cancer",
    "Myocardial infarction (disorder)": "Myocardial Infarction",
    "Non-small cell lung cancer (disorder)": "Non-small Cell Lung Cancer",
    "Seizure disorder": "Seizure Disorder",
    "Suspected lung cancer (situation)": "Suspected Lung Cancer",
}
df["Disease Name"] = df["Disease Name"].map(name_map).fillna(df["Disease Name"])

# Flag IoT endpoints (as used in your figure labels)
df["DeviceLabel"] = df["current_device"].replace(
    {"Device3": "Device3 (IoT device)", "Device26": "Device26 (IoT device)"}
)

df["is_IoT"] = df["current_device"].isin(["Device3", "Device26"])

# Order devices numerically
def dev_key(s):
    m = re.search(r"(\d+)", str(s))
    return int(m.group(1)) if m else 10**9

# --------------------------
# 2) Helper to compute robust stats
# --------------------------
def summarize(series):
    x = series.to_numpy()
    n = x.size
    if n == 0:
        return dict(N=0, median=np.nan, q1=np.nan, q3=np.nan, iqr=np.nan,
                    p90=np.nan, p95=np.nan, mean=np.nan, sd=np.nan, frac_le_0p2=np.nan)
    q1, q3 = np.percentile(x, [25, 75])
    return dict(
        N=n,
        median=float(np.median(x)),
        q1=float(q1),
        q3=float(q3),
        iqr=float(q3 - q1),
        p90=float(np.percentile(x, 90)),
        p95=float(np.percentile(x, 95)),
        mean=float(np.mean(x)),
        sd=float(np.std(x, ddof=1)) if n > 1 else 0.0,
        frac_le_0p2=float((x <= 0.2).mean())
    )

# --------------------------
# 3) Stats by Device × Disease
# --------------------------
rows = []
for (dev, dis), sub in df.groupby(["DeviceLabel", "Disease Name"]):
    stats = summarize(sub["Time per Device (t_dash)"])
    stats.update({"Device": dev, "Disease": dis})
    rows.append(stats)
by_dev_dis = pd.DataFrame(rows).sort_values(["Device"], key=lambda s: s.map(dev_key)).reset_index(drop=True)
by_dev_dis.to_csv("device_disease_metrics.csv", index=False)

# --------------------------
# 4) Stats by Device (overall across diseases)
# --------------------------
rows = []
for dev, sub in df.groupby("DeviceLabel"):
    stats = summarize(sub["Time per Device (t_dash)"])
    stats.update({"Device": dev, "is_IoT": bool(sub["is_IoT"].iloc[0])})
    rows.append(stats)
by_device = pd.DataFrame(rows).sort_values(["Device"], key=lambda s: s.map(dev_key)).reset_index(drop=True)
by_device.to_csv("device_metrics_overall.csv", index=False)

# --------------------------
# 5) Stats by Disease (overall across devices)
# --------------------------
rows = []
for dis, sub in df.groupby("Disease Name"):
    stats = summarize(sub["Time per Device (t_dash)"])
    stats.update({"Disease": dis})
    rows.append(stats)
by_disease = pd.DataFrame(rows).sort_values("Disease").reset_index(drop=True)
by_disease.to_csv("disease_metrics_overall.csv", index=False)

# --------------------------
# 6) Auto-generate concise highlights (txt)
# --------------------------
def fmt_range(q1, q3):
    if np.isnan(q1) or np.isnan(q3):
        return "NA"
    return f"{q1:.2f}–{q3:.2f}s"

lines = []
lines.append("=== Device performance highlights ===")

# IoT devices
for dev in ["Device3 (IoT device)", "Device26 (IoT device)"]:
    row = by_device[by_device["Device"] == dev]
    if len(row):
        r = row.iloc[0]
        lines.append(
            f"{dev}: median {r['median']:.2f}s, IQR {r['iqr']:.2f}s "
            f"({fmt_range(r['q1'], r['q3'])}), P90 {r['p90']:.2f}s, N={int(r['N'])}"
        )

# Mid-tier devices: show typical range of medians across diseases
mid = by_device[~by_device["Device"].isin(["Device3 (IoT device)", "Device26 (IoT device)"])]
if len(mid):
    lines.append(
        f"Mid-tier devices (non-IoT): median across all = {mid['median'].median():.2f}s "
        f"(IQR of device medians {mid['median'].quantile(0.25):.2f}–{mid['median'].quantile(0.75):.2f}s)."
    )

# Per disease quick snapshot (share ≤0.2s)
dis_snaps = []
for _, r in by_disease.iterrows():
    dis_snaps.append(f"{r['Disease']}: median {r['median']:.2f}s, ≤0.2s share {100*r['frac_le_0p2']:.1f}%")
lines.append("By disease (overall across devices): " + "; ".join(dis_snaps))

with open("device_perf_highlights.txt", "w") as f:
    f.write("\n".join(lines))

print("\n".join(lines))
print("\nWrote:")
print(" - device_disease_metrics.csv")
print(" - device_metrics_overall.csv")
print(" - disease_metrics_overall.csv")
print(" - device_perf_highlights.txt")


## Section 6: Resource Consumption Metrics (CPU, Memory, RAM)

Compute 95% CI resource consumption (CPU %, memory MB, RAM MB) per device and disease. Two versions:
1. Raw percentages with error bars
2. Normalized CPU fraction domain [0,1] with bounded CIs

Outputs: `resource_metrics_by_device_*.csv` files and paper figures.

In [ ]:
#!/usr/bin/env python3
# Resource consumption metrics + plots (with 95% CI error bars)

import pandas as pd
import numpy as np
import re
from scipy.stats import t
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

IN = "results.csv"

# ----------------------------
# Load & clean
# ----------------------------
use_cols = ["current_device", "Disease Name", "CPU Usage (%)", "Memory Usage (MB)", "RAM Usage (MB)"]
df = pd.read_csv(IN, usecols=lambda c: c in use_cols)

# numeric coercion
for c in ["CPU Usage (%)", "Memory Usage (MB)", "RAM Usage (MB)"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# drop missing
df = df.dropna(subset=["current_device", "Disease Name", "CPU Usage (%)", "Memory Usage (MB)", "RAM Usage (MB)"]).copy()

# normalize disease names
name_map = {
    "Myocardialinfarction(disorder)": "Myocardial Infarction",
    "Non-smallcelllungcancer(disorder)": "Non-small Cell Lung Cancer",
    "Seizuredisorder": "Seizure Disorder",
    "Stroke": "Stroke",
    "Suspectedlungcancer(situation)": "Suspected Lung Cancer",
    "Myocardial infarction (disorder)": "Myocardial Infarction",
    "Non-small cell lung cancer (disorder)": "Non-small Cell Lung Cancer",
    "Seizure disorder": "Seizure Disorder",
    "Suspected lung cancer (situation)": "Suspected Lung Cancer",
}
df["Disease Name"] = df["Disease Name"].map(name_map).fillna(df["Disease Name"])

# label IoT devices (for x-axis text)
df["current_device"] = df["current_device"].replace(
    {"Device3": "Device3\n(IoT device)", "Device26": "Device26\n(IoT device)"}
)


# numeric device sort
def dev_key(s):
    m = re.search(r"(\d+)", str(s))
    return int(m.group(1)) if m else 10**9

sorted_devices = sorted(df["current_device"].unique(), key=dev_key)

# ----------------------------
# Stats helper: mean ± 95% CI (t)
# ----------------------------
def mean_ci(series):
    x = series.to_numpy(dtype=float)
    n = x.size
    if n == 0:
        return np.nan, np.nan
    m = float(np.mean(x))
    if n == 1:
        return m, 0.0
    s = float(np.std(x, ddof=1))
    half = t.ppf(0.975, n-1) * s / np.sqrt(n)
    return m, half

# ----------------------------
# Compute per (device, disease)
# ----------------------------
metrics = ["CPU Usage (%)", "Memory Usage (MB)", "RAM Usage (MB)"]
rows = []
for (dev, disease), g in df.groupby(["current_device", "Disease Name"]):
    rec = {"Device": dev, "Disease": disease, "N": len(g)}
    for m in metrics:
        mean, half = mean_ci(g[m])
        rec[f"{m} mean"] = mean
        rec[f"{m} ci95"] = half
    rows.append(rec)
by_dev_dis = pd.DataFrame(rows).sort_values(["Device", "Disease"], key=lambda s: s.map(dev_key) if s.name=="Device" else s).reset_index(drop=True)
by_dev_dis.to_csv("resource_metrics_by_device_disease.csv", index=False)

# Also overall by device (across diseases)
rows = []
for dev, g in df.groupby("current_device"):
    rec = {"Device": dev, "N": len(g)}
    for m in metrics:
        mean, half = mean_ci(g[m])
        rec[f"{m} mean"] = mean
        rec[f"{m} ci95"] = half
    rows.append(rec)
by_device = pd.DataFrame(rows).sort_values("Device", key=lambda s: s.map(dev_key)).reset_index(drop=True)
by_device.to_csv("resource_metrics_by_device_overall.csv", index=False)

# And overall by disease (across devices)
rows = []
for disease, g in df.groupby("Disease Name"):
    rec = {"Disease": disease, "N": len(g)}
    for m in metrics:
        mean, half = mean_ci(g[m])
        rec[f"{m} mean"] = mean
        rec[f"{m} ci95"] = half
    rows.append(rec)
by_disease = pd.DataFrame(rows).sort_values("Disease").reset_index(drop=True)
by_disease.to_csv("resource_metrics_by_disease_overall.csv", index=False)

print("[OK] Wrote CSVs:",
      "resource_metrics_by_device_disease.csv,",
      "resource_metrics_by_device_overall.csv,",
      "resource_metrics_by_disease_overall.csv")

# ----------------------------
# Plotting (multi-bar with 95% CI)
# ----------------------------
disease_styles = {
    "Myocardial Infarction":        {'color': 'lightblue',  'hatch': '/'},
    "Non-small Cell Lung Cancer":   {'color': 'lightgreen', 'hatch': '*'},
    "Seizure Disorder":             {'color': 'lightpink',  'hatch': '|'},
    "Stroke":                       {'color': 'lightyellow','hatch': '-'},
    "Suspected Lung Cancer":        {'color': 'lightcoral', 'hatch': '+'},
}
font0 = {'family': 'Times New Roman', 'size': 30}
font1 = {'family': 'Times New Roman', 'size': 32}
font2 = {'family': 'Times New Roman', 'size': 32}
font3 = {'family': 'Times New Roman', 'size': 26}

from matplotlib.patches import Patch
def plot_with_ci(df_stats, value_col, ci_col, y_label, file_name,
                 legend_position="upper right", bbox_to_anchor=None):
    # pivot into mean and CI matrices
    mean_p = df_stats.pivot(index="Device", columns="Disease", values=value_col).reindex(sorted_devices)
    ci_p   = df_stats.pivot(index="Device", columns="Disease", values=ci_col).reindex(sorted_devices)

    devices = mean_p.index
    diseases = mean_p.columns
    x = np.arange(len(devices))
    w = 0.15

    plt.figure(figsize=(max(12, len(devices)*1.5), 10))
    for i, disease in enumerate(diseases):
        means = mean_p[disease].values
        errs  = ci_p[disease].values
        plt.bar(
            x + i*w, means, width=w,
            color=disease_styles[disease]['color'],
            hatch=disease_styles[disease]['hatch'],
            edgecolor='black',
            label=disease if i==0 else None,
            yerr=errs, capsize=5, ecolor='black', error_kw={"elinewidth":1.2}
        )

    legend_handles = [
        Patch(facecolor=style['color'], edgecolor='black', hatch=style['hatch'], label=disease)
        for disease, style in disease_styles.items()
        if disease in diseases
    ]
    plt.legend(handles=legend_handles, loc=legend_position, bbox_to_anchor=bbox_to_anchor, fontsize=font3['size'])
    plt.xticks(x + (len(diseases)-1)*w/2, devices, rotation=45, ha='center', **font2)
    plt.yticks(**font2)
    plt.xlabel("Device Name", **font1)
    plt.ylabel(y_label, **font1)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(file_name, bbox_inches="tight", dpi=300)
    plt.show()

# call plots using the per (device, disease) stats we computed
plot_with_ci(by_dev_dis, "RAM Usage (MB) mean",    "RAM Usage (MB) ci95",    "RAM Usage (MB)",    "ram_usage_multibar_ci.pdf",    "upper right")
plot_with_ci(by_dev_dis, "CPU Usage (%) mean",     "CPU Usage (%) ci95",     "CPU Usage (%)",     "cpu_usage_multibar_ci.pdf",    "upper right")
plot_with_ci(by_dev_dis, "Memory Usage (MB) mean", "Memory Usage (MB) ci95", "Memory Usage (MB)", "memory_usage_multibar_ci.pdf", "upper center")

# ----------------------------
# Optional: print a few headline numbers for writing
# ----------------------------
def headline(df_stats, metric):
    mcol, ccol = f"{metric} mean", f"{metric} ci95"
    # IoT devices overall (across diseases)
    for dev in ["Device3\n(IoT device)", "Device26\n(IoT device)"]:
        sub = df_stats[df_stats["Device"]==dev]
        if len(sub):
            print(f"{dev} – {metric}: mean across diseases = "
                  f"{sub[mcol].mean():.1f} ± {sub[ccol].mean():.1f}")

print("\nHeadlines:")
for met in ["RAM Usage (MB)", "CPU Usage (%)", "Memory Usage (MB)"]:
    headline(by_dev_dis, met)


In [ ]:
#!/usr/bin/env python3
# Resource consumption metrics + plots (CPU in FRACTION domain [0,1], bounded 95% CIs)
# - Drops any rows where CPU Usage (%) > 100 (pre-normalization) or > 1.0 after normalization.

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter

IN = "results.csv"

# ----------------------------
# Load & clean
# ----------------------------
use_cols = ["current_device", "Disease Name",
            "CPU Usage (%)", "Memory Usage (MB)", "RAM Usage (MB)"]
df = pd.read_csv(IN, usecols=lambda c: c in use_cols)

# numeric coercion
for c in ["CPU Usage (%)", "Memory Usage (MB)", "RAM Usage (MB)"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# drop missing
df = df.dropna(subset=["current_device", "Disease Name",
                       "CPU Usage (%)", "Memory Usage (MB)", "RAM Usage (MB)"]).copy()

# remove negatives
for c in ["CPU Usage (%)", "Memory Usage (MB)", "RAM Usage (MB)"]:
    df = df[df[c] >= 0]

# --- REMOVE CPU > 100% rows BEFORE normalization ---
before_rows = len(df)
df = df[df["CPU Usage (%)"] <= 100]
removed_pre_norm = before_rows - len(df)

# --- FORCE CPU TO FRACTION [0,1] ---
cpu_max = df["CPU Usage (%)"].max()
if pd.notna(cpu_max) and cpu_max > 1.5:
    # values are in percent -> convert to fraction
    df["CPU Usage (%)"] = df["CPU Usage (%)"] / 100.0

# --- REMOVE CPU > 1.0 rows AFTER normalization (just in case) ---
before_rows2 = len(df)
df = df[df["CPU Usage (%)"] <= 1.0]
removed_post_norm = before_rows2 - len(df)

# Clamp to [0,1] (safety)
df["CPU Usage (%)"] = df["CPU Usage (%)"].clip(lower=0.0, upper=1.0)

# normalize disease names
name_map = {
    "Myocardialinfarction(disorder)": "Myocardial Infarction",
    "Non-smallcelllungcancer(disorder)": "Non-small Cell Lung Cancer",
    "Seizuredisorder": "Seizure Disorder",
    "Stroke": "Stroke",
    "Suspectedlungcancer(situation)": "Suspected Lung Cancer",
    "Myocardial infarction (disorder)": "Myocardial Infarction",
    "Non-small cell lung cancer (disorder)": "Non-small Cell Lung Cancer",
    "Seizure disorder": "Seizure Disorder",
    "Suspected lung cancer (situation)": "Suspected Lung Cancer",
}
df["Disease Name"] = df["Disease Name"].map(name_map).fillna(df["Disease Name"])

# label IoT devices
df["current_device"] = df["current_device"].replace(
    {"Device3": "Device3\n(IoT device)", "Device26": "Device26\n(IoT device)"}
)

# numeric device sort
def dev_key(s):
    m = re.search(r"(\d+)", str(s))
    return int(m.group(1)) if m else 10**9
sorted_devices = sorted(df["current_device"].unique(), key=dev_key)

# ----------------------------
# Bounded bootstrap mean CIs (asymmetric)
# ----------------------------
def bootstrap_mean_ci_bounded(x, lower=0.0, upper=None, B=3000, alpha=0.05, seed=0):
    v = np.asarray(x, float)
    v = v[np.isfinite(v)]
    n = v.size
    if n == 0: return np.nan, np.nan, np.nan
    m = float(np.mean(v))
    if n == 1:
        lo = hi = m
    else:
        rng = np.random.default_rng(seed)
        boots = [np.mean(v[rng.integers(0, n, n)]) for _ in range(B)]
        lo, hi = np.percentile(boots, [100*alpha/2, 100*(1-alpha/2)])
    if lower is not None:
        lo = max(lower, lo); hi = max(lower, hi)
    if upper is not None:
        lo = min(upper, lo); hi = min(upper, hi)
    lo = min(lo, m); hi = max(hi, m)
    return m, lo, hi

# ----------------------------
# Compute per (device, disease)
# ----------------------------
metrics_info = {
    "CPU Usage (%)":      {"lower": 0.0, "upper": 1.0},   # FRACTION domain
    "Memory Usage (MB)":  {"lower": 0.0, "upper": None},
    "RAM Usage (MB)":     {"lower": 0.0, "upper": None},
}

rows = []
for (dev, disease), g in df.groupby(["current_device", "Disease Name"]):
    rec = {"Device": dev, "Disease": disease, "N": len(g)}
    for m, bounds in metrics_info.items():
        mean, lo, hi = bootstrap_mean_ci_bounded(
            g[m], lower=bounds["lower"], upper=bounds["upper"], B=3000, seed=42
        )
        rec[f"{m} mean"] = mean
        rec[f"{m} lo"]   = lo
        rec[f"{m} hi"]   = hi
    rows.append(rec)
by_dev_dis = (pd.DataFrame(rows)
                .sort_values(["Device","Disease"],
                             key=lambda s: s.map(dev_key) if s.name=="Device" else s)
                .reset_index(drop=True))
by_dev_dis.to_csv("resource_metrics_by_device_disease.csv", index=False)

# overall by device
rows = []
for dev, g in df.groupby("current_device"):
    rec = {"Device": dev, "N": len(g)}
    for m, bounds in metrics_info.items():
        mean, lo, hi = bootstrap_mean_ci_bounded(
            g[m], lower=bounds["lower"], upper=bounds["upper"], B=3000, seed=123
        )
        rec[f"{m} mean"] = mean
        rec[f"{m} lo"]   = lo
        rec[f"{m} hi"]   = hi
    rows.append(rec)
by_device = pd.DataFrame(rows).sort_values("Device", key=lambda s: s.map(dev_key)).reset_index(drop=True)
by_device.to_csv("resource_metrics_by_device_overall.csv", index=False)

# overall by disease
rows = []
for disease, g in df.groupby("Disease Name"):
    rec = {"Disease": disease, "N": len(g)}
    for m, bounds in metrics_info.items():
        mean, lo, hi = bootstrap_mean_ci_bounded(
            g[m], lower=bounds["lower"], upper=bounds["upper"], B=3000, seed=321
        )
        rec[f"{m} mean"] = mean
        rec[f"{m} lo"]   = lo
        rec[f"{m} hi"]   = hi
    rows.append(rec)
by_disease = pd.DataFrame(rows).sort_values("Disease").reset_index(drop=True)
by_disease.to_csv("resource_metrics_by_disease_overall.csv", index=False)

print("[OK] Wrote CSVs:",
      "resource_metrics_by_device_disease.csv,",
      "resource_metrics_by_device_overall.csv,",
      "resource_metrics_by_disease_overall.csv")
print(f"Removed rows with CPU > 100% (pre-norm): {removed_pre_norm}, "
      f"and CPU > 1.0 (post-norm): {removed_post_norm}.")
print("CPU scale normalized to FRACTION [0,1].")

# ----------------------------
# Plotting (CPU in fraction [0,1]; tick labels as %)
# ----------------------------
disease_styles = {
    "Myocardial Infarction":        {'color': 'lightblue',  'hatch': '/'},
    "Non-small Cell Lung Cancer":   {'color': 'lightgreen', 'hatch': '*'},
    "Seizure Disorder":             {'color': 'lightpink',  'hatch': '|'},
    "Stroke":                       {'color': 'lightyellow','hatch': '-'},
    "Suspected Lung Cancer":        {'color': 'lightcoral', 'hatch': '+'},
}
font1 = {'family': 'Times New Roman', 'size': 32}
font2 = {'family': 'Times New Roman', 'size': 32}
font3 = {'family': 'Times New Roman', 'size': 30}

def plot_with_asym_ci(df_stats, mean_col, lo_col, hi_col, y_label, file_name,
                      legend_position="upper right", bbox_to_anchor=None,
                      lower=None, upper=None, percent_ticks=False, y_max_fraction=None):
    mean_p = df_stats.pivot(index="Device", columns="Disease", values=mean_col).reindex(sorted_devices)
    lo_p   = df_stats.pivot(index="Device", columns="Disease", values=lo_col).reindex(sorted_devices)
    hi_p   = df_stats.pivot(index="Device", columns="Disease", values=hi_col).reindex(sorted_devices)

    devices  = mean_p.index
    diseases = mean_p.columns
    if len(diseases) == 0:
        print(f"[WARN] No non-NaN data for {y_label}; skipping plot {file_name}.")
        return

    x = np.arange(len(devices))
    w = 0.15

    plt.figure(figsize=(max(12, len(devices)*1.5), 10))
    for i, disease in enumerate(diseases):
        means = mean_p[disease].to_numpy(float)
        los   = lo_p[disease].to_numpy(float)
        his   = hi_p[disease].to_numpy(float)

        if lower is not None:
            means = np.maximum(means, lower); los = np.maximum(los, lower); his = np.maximum(his, lower)
        if upper is not None:
            means = np.minimum(means, upper);  los = np.minimum(los, upper);  his = np.minimum(his, upper)

        down = np.clip(means - los, 0, None)
        up   = np.clip(his - means, 0, None)
        yerr = np.vstack([down, up])

        plt.bar(
            x + i*w, means, width=w,
            color=disease_styles[disease]['color'],
            hatch=disease_styles[disease]['hatch'],
            edgecolor='black',
            label=disease if i==0 else None,
            yerr=yerr, capsize=5, ecolor='black', error_kw={"elinewidth":1.2}
        )

    legend_handles = [Patch(facecolor=style['color'], edgecolor='black',
                            hatch=style['hatch'], label=disease)
                      for disease, style in disease_styles.items() if disease in diseases]
    plt.legend(handles=legend_handles, loc=legend_position,
               bbox_to_anchor=bbox_to_anchor, fontsize=font3['size'])
    plt.xticks(x + (len(diseases)-1)*w/2, devices, rotation=45, ha='center', **font2)
    plt.xlabel("Device Name", **font1)
    plt.ylabel(y_label, **font1)
    plt.gca().tick_params(axis='y', labelsize=30)


    # ---- Dynamic Y range from tallest bar + CI ----
    if percent_ticks:
        ax = plt.gca()
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))  # fraction→percent
        data_ymax = np.nanmax(hi_p.to_numpy(dtype=float))
        ymax = min(1.0, max(0.01, data_ymax) * 1.05)  # 5% headroom, max 100%
        plt.ylim(0, ymax)
    else:
        data_ymax = np.nanmax(hi_p.to_numpy(dtype=float))
        ymax = max(1.0, (data_ymax if np.isfinite(data_ymax) else 1.0) * 1.05)
        plt.ylim(0, ymax)

    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(file_name, bbox_inches="tight", dpi=300)
    plt.show()

# RAM / Memory (nonnegative)
plot_with_asym_ci(by_dev_dis, "RAM Usage (MB) mean", "RAM Usage (MB) lo", "RAM Usage (MB) hi",
                  "RAM Usage (MB)", "ram_usage_multibar_ci.pdf", "upper right",
                  lower=0.0, upper=None, percent_ticks=False)

plot_with_asym_ci(by_dev_dis, "Memory Usage (MB) mean", "Memory Usage (MB) lo", "Memory Usage (MB) hi",
                  "Memory Usage (MB)", "memory_usage_multibar_ci.pdf", "upper center",
                  lower=0.0, upper=None, percent_ticks=False)

# CPU in FRACTION with percent ticks; y-axis auto-scales to tallest bar + CI
plot_with_asym_ci(by_dev_dis, "CPU Usage (%) mean", "CPU Usage (%) lo", "CPU Usage (%) hi",
                  "CPU Usage (%)", "cpu_usage_multibar_ci.pdf", "upper right",
                  lower=0.0, upper=1.0, percent_ticks=True, y_max_fraction=None)


## Section 7: Device Usage Metrics

Compute per-disease and global device participation statistics from Chain Data. Measures how frequently each device appears in chains across diseases.

Outputs: `device_usage_metrics.csv`, `device_usage_global.csv`, `device_usage_paragraph.tex`

In [ ]:
#!/usr/bin/env python3
# Build quantitative device-usage metrics CSVs + LaTeX paragraph
# Inputs: results.csv  (expects columns: ["Disease Name","Chain Data","Total Time (T)"])
# Outputs:
#   - device_usage_metrics.csv   (per-disease metrics)
#   - device_usage_global.csv    (overall device shares)
#   - device_usage_paragraph.tex (ready-to-include LaTeX paragraph)

import ast, numpy as np, pandas as pd
from math import log2

RESULTS_CSV = "results.csv"

# ---- normalize disease names (edit as needed) ----
NAME_MAP = {
    "Myocardialinfarction(disorder)": "Myocardial Infarction",
    "Non-smallcelllungcancer(disorder)": "Non-small Cell Lung Cancer",
    "Seizuredisorder": "Seizure Disorder",
    "Stroke": "Stroke",
    "Suspectedlungcancer(situation)": "Suspected Lung Cancer",
    "Myocardial infarction (disorder)": "Myocardial Infarction",
    "Non-small cell lung cancer (disorder)": "Non-small Cell Lung Cancer",
    "Suspected lung cancer (situation)": "Suspected Lung Cancer",
    "Seizure disorder": "Seizure Disorder",
}

def parse_devices(chain_data):
    """Each Chain Data entry is a list/tuple; device name at index 4."""
    try:
        lst = ast.literal_eval(str(chain_data))
        return [e[4] for e in lst if isinstance(e, (list, tuple)) and len(e) > 4]
    except Exception:
        return []

def shannon_entropy(counts):
    tot = counts.sum()
    if tot == 0: return 0.0
    p = counts / tot
    return float(-(p * np.log2(np.clip(p, 1e-12, 1))).sum())

def hhi(counts):
    tot = counts.sum()
    if tot == 0: return 0.0
    p = counts / tot
    return float((p**2).sum())

# ---- load & clean ----
df = pd.read_csv(RESULTS_CSV)
if "Disease Name" not in df.columns or "Chain Data" not in df.columns:
    raise SystemExit("results.csv must contain 'Disease Name' and 'Chain Data' columns.")
df["Disease Name"] = df["Disease Name"].map(NAME_MAP).fillna(df["Disease Name"])
df["Chain Data"] = df["Chain Data"].astype(str).replace({"inf":"1e9"}, regex=False)
if "Total Time (T)" in df.columns:
    df = df[df["Total Time (T)"] > 0].copy()

# ---- extract device occurrences per row ----
df["DevicesInPath"] = df["Chain Data"].apply(parse_devices)

# build long form: (Disease, Device) per occurrence
rows = []
for disease, sub in df.groupby("Disease Name"):
    for devs in sub["DevicesInPath"]:
        for dev in devs:
            rows.append((disease, dev))
usage = pd.DataFrame(rows, columns=["Disease","Device"])
if usage.empty:
    raise SystemExit("No device occurrences could be parsed from Chain Data.")

# ---- per-disease metrics ----
records = []
for disease, grp in usage.groupby("Disease"):
    counts = grp["Device"].value_counts().sort_values(ascending=False)
    total  = int(counts.sum())
    devices_used = int(counts.size)
    top1_dev = counts.index[0]
    top1 = int(counts.iloc[0])
    top2 = int(counts.iloc[:2].sum()) if devices_used >= 2 else top1
    rec = dict(
        Disease=disease,
        TotalOccurrences=total,
        DevicesUsed=devices_used,
        TopDevice=top1_dev,
        Top1Count=top1,
        Top1SharePct=round(100*top1/total, 1) if total else 0.0,
        Top2SharePct=round(100*top2/total, 1) if total else 0.0,
        HHI=round(hhi(counts), 3),
        Entropy=round(shannon_entropy(counts), 2),
    )
    records.append(rec)

per_disease = pd.DataFrame(records).sort_values("Disease").reset_index(drop=True)
per_disease.to_csv("device_usage_metrics.csv", index=False)

# ---- global device shares ----
global_counts = usage["Device"].value_counts()
global_total  = int(global_counts.sum())
global_df = pd.DataFrame({
    "Device": global_counts.index,
    "Count": global_counts.values,
    "SharePct": (100*global_counts.values/global_total).round(1)
})
global_df.to_csv("device_usage_global.csv", index=False)

# ---- write LaTeX paragraph with numbers plugged in ----
top_global_dev = global_df.iloc[0]["Device"]
top_global_share = global_df.iloc[0]["SharePct"]
lines = []
lines.append("\\subsubsection{Device usage distribution across diseases (CasFly)}")
lines.append("\\textbf{Why this analysis.} Understanding which devices are most frequently traversed by LaVE helps budget compute, plan redundancy, and reason about privacy-preserving specialization at the edge.")
lines.append("")
lines.append("Figure~\\ref{fig:variationdevice} summarizes empirical device usage extracted from LaVE chains. "
             f"Across all diseases, \\textbf{{{top_global_dev}}} accounts for \\textbf{{{top_global_share}\\%}} of all device occurrences, "
             "indicating a shared hub role in the federation.")

for _, r in per_disease.iterrows():
    lines.append((
        f"\\emph{{{r['Disease']}}} involves \\textbf{{{r['DevicesUsed']}}} devices (total occurrences: {r['TotalOccurrences']}). "
        f"The most frequent device is \\textbf{{{r['TopDevice']}}} "
        f"(top-1 share \\textbf{{{r['Top1SharePct']}\\%}}, top-2 share \\textbf{{{r['Top2SharePct']}\\%}}). "
        f"Concentration metrics: HHI=\\textbf{{{r['HHI']}}}, Entropy=\\textbf{{{r['Entropy']}}}."))
# Add a closing sentence
lines.append("These quantitative profiles confirm that certain devices (e.g., Device20/Device7) act as pivotal hubs, "
             "whereas others (e.g., Device4/Device16) contribute sparsely. The concentration statistics (Top-$k$ shares, HHI/Entropy) "
             "are actionable for capacity planning, fault tolerance (replicating hubs), and privacy-aware model placement at the edge.")

with open("device_usage_paragraph.tex","w") as f:
    f.write("\n\n".join(lines))

print("Wrote:")
print(" - device_usage_metrics.csv")
print(" - device_usage_global.csv")
print(" - device_usage_paragraph.tex")


## Section 8: Device Storage Size Visualization (Fig 6)

Read TPHG pickle sizes and CSV sizes from `device_partitions_patientwise1_rasp/`. Plot per-device storage footprint to compare TPHG vs flat-CSV storage.

In [ ]:
import os
import matplotlib.pyplot as plt



font0 = {'family': 'Times New Roman', 'size': 28}
font1 = {'family': 'Times New Roman', 'size': 32}
font2 = {'family': 'Times New Roman', 'size': 30}
font3 = {'family': 'Times New Roman', 'size': 33}


base_directory = "device_partitions_patientwise1_rasp"


devices_to_consider = ["Device7", "Device26", "Device3", "Device11", "Device5", "Device14", "Device4", "Device13", "Device16"]


device_names = []
max_tphg_sizes = []
csv_sizes = []


for device in devices_to_consider:
    device_folder = os.path.join(base_directory, device)
    if os.path.exists(device_folder):
        max_tphg_size = 0
        csv_file_size = 0

        
        for file_name in os.listdir(device_folder):
            file_path = os.path.join(device_folder, file_name)
            if file_name.endswith("_tphg.pkl"):
                max_tphg_size = max(max_tphg_size, os.path.getsize(file_path))
            elif file_name.endswith("causal_pairs.csv"):
                csv_file_size = os.path.getsize(file_path)

        if max_tphg_size > 0 and csv_file_size > 0:
            device_names.append(device)
            max_tphg_sizes.append(max_tphg_size / (1024 * 1024))  
            csv_sizes.append(csv_file_size / (1024 * 1024))  


def sort_device_key(device):
    return int(''.join(filter(str.isdigit, device)))

sorted_indices = sorted(range(len(device_names)), key=lambda i: sort_device_key(device_names[i]))
device_names = [device_names[i] for i in sorted_indices]
max_tphg_sizes = [max_tphg_sizes[i] for i in sorted_indices]
csv_sizes = [csv_sizes[i] for i in sorted_indices]


mean_max_tphg = sum(max_tphg_sizes) / len(max_tphg_sizes)
mean_csv_size = sum(csv_sizes) / len(csv_sizes)
percentage_reduction = ((mean_csv_size - mean_max_tphg) / mean_csv_size) * 100


plt.figure(figsize=(14, 8))
x = range(len(device_names))

plt.bar(x, max_tphg_sizes, width=0.4, label='Max TPHG Size (MB)', align='center', hatch='//', color='skyblue', edgecolor='black')
plt.bar([i + 0.4 for i in x], csv_sizes, width=0.4, label='Conditional Probability Table (MB)', align='center', hatch='\\', color='lightgreen', edgecolor='black')


plt.yscale('log')

plt.xlabel('Device Names', fontsize=font1['size'], family=font1['family'])
plt.ylabel('Size (MB, Log Scale)', fontsize=font1['size'], family=font1['family'])
plt.xticks([i + 0.2 for i in x], device_names, rotation=45, ha='right', fontsize=font2['size'], family=font2['family'])
plt.legend(fontsize=font0['size'],  loc="upper right")
plt.yticks(fontsize=font2['size'])
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('cptwithtphgstorage.png')
plt.show()

print(f"Mean Max TPHG Size: {mean_max_tphg:.2f} MB")
print(f"Mean CSV Size: {mean_csv_size:.2f} MB")
print(f"Percentage Reduction in Size: {percentage_reduction:.2f}%")


## Section 9: CPT vs TPHG Load Time and Storage Comparison (Fig 6)

Unified script: reads `device_metrics_analysiscptloadtime.csv` and `results.csv`, computes load-time statistics with 95% CI whiskers, and regenerates the storage comparison figure.

Outputs: `cpt_vs_tphg_load_time_comparison_log.png`, `cptwithtphgstorage.png`, `metrics_summary.txt`

In [ ]:
#!/usr/bin/env python3
# One script: compute metrics + regenerate both figures
# - Reads:
#     device_metrics_analysiscptloadtime.csv  (CPT load-time rows)
#     results.csv                             (TPHG load-time rows)
#     device_partitions_patientwise1_rasp/    (per-device sizes)
# - Writes:
#     cpt_vs_tphg_load_time_comparison_log.png  (with 95% CI whiskers)
#     cptwithtphgstorage.png                    (storage comparison)
#     metrics_summary.txt                       (human-readable report)

import os, sys, numpy as np, pandas as pd
import matplotlib.pyplot as plt

# -------------------------------
# Paths / config
# -------------------------------
BASE_DIR = "device_partitions_patientwise1_rasp"
CPT_CSV  = "device_metrics_analysiscptloadtime.csv"  # cols: Device, Load Time (s)
RES_CSV  = "results.csv"                             # cols: current_device, Cached TPHG Load Time (t1)

# If you want to restrict devices, list them here; otherwise use all detected:
DEVICES_TO_CONSIDER = None  # e.g., ["Device7","Device26","Device3","Device11","Device5","Device14","Device4","Device13","Device16"]

# Fonts (match your paper style)
font0 = {'family': 'Times New Roman', 'size': 28}
font1 = {'family': 'Times New Roman', 'size': 32}
font2 = {'family': 'Times New Roman', 'size': 30}
font3 = {'family': 'Times New Roman', 'size': 33}

# -------------------------------
# Helpers
# -------------------------------
def dev_key(d):  # numeric sort by device suffix
    try:
        return int(''.join(filter(str.isdigit, str(d))))
    except:
        return 10**9

def bootstrap_mean_ci(x, B=5000, alpha=0.05, seed=0):
    """Bootstrap 95% CI of the mean. Returns (mean, lo, hi)."""
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    if len(x) == 1:
        m = float(x[0])
        return m, m, m
    rng = np.random.default_rng(seed)
    n = len(x)
    boots = []
    for _ in range(B):
        samp = x[rng.integers(0, n, n)]
        boots.append(np.mean(samp))
    lo, hi = np.percentile(boots, [100*alpha/2, 100*(1-alpha/2)])
    return float(np.mean(x)), float(lo), float(hi)

def gmean(vals):
    v = np.asarray(vals, dtype=float)
    v = v[v > 0]
    if len(v) == 0:
        return np.nan
    return float(np.exp(np.mean(np.log(v))))

def boot_ci(vals, fn=np.mean, B=5000, alpha=0.05, seed=0):
    """Generic bootstrap CI for a functional fn over vals."""
    v = np.asarray(vals, dtype=float)
    if len(v) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    n = len(v)
    boots = []
    for _ in range(B):
        samp = v[rng.integers(0, n, n)]
        boots.append(fn(samp))
    lo, hi = np.percentile(boots, [100*alpha/2, 100*(1-alpha/2)])
    return float(lo), float(hi)

def collect_sizes(base_dir, allow_devices=None):
    rows = []
    if not os.path.isdir(base_dir):
        return pd.DataFrame(columns=["Device","TPHG_Size_MB","CPT_Size_MB"])
    for dev in sorted(os.listdir(base_dir), key=dev_key):
        if allow_devices and dev not in allow_devices:
            continue
        dev_dir = os.path.join(base_dir, dev)
        if not os.path.isdir(dev_dir):
            continue
        max_tphg = 0
        cpt_sz = 0
        for fn in os.listdir(dev_dir):
            fp = os.path.join(dev_dir, fn)
            if fn.endswith("_tphg.pkl"):
                try:
                    max_tphg = max(max_tphg, os.path.getsize(fp))
                except OSError:
                    pass
            elif fn.endswith("causal_pairs.csv"):
                try:
                    cpt_sz = os.path.getsize(fp)
                except OSError:
                    pass
        if max_tphg > 0 and cpt_sz > 0:
            rows.append({
                "Device": dev,
                "TPHG_Size_MB": max_tphg/(1024*1024),
                "CPT_Size_MB":  cpt_sz/(1024*1024),
            })
    return pd.DataFrame(rows)

# -------------------------------
# Load data
# -------------------------------
if not os.path.exists(CPT_CSV) or not os.path.exists(RES_CSV):
    sys.exit(f"Missing CSVs. Expected: {CPT_CSV} and {RES_CSV}")

cpt_df = pd.read_csv(CPT_CSV)
res_df = pd.read_csv(RES_CSV)

# Determine device universe
devices = sorted(
    set(cpt_df["Device"]).intersection(set(res_df["current_device"])),
    key=dev_key
)
if DEVICES_TO_CONSIDER:
    devices = [d for d in devices if d in DEVICES_TO_CONSIDER]

if len(devices) == 0:
    sys.exit("No common devices found between the two CSVs.")

# -------------------------------
# Per-device load-time means + 95% CIs (bootstrap)
# -------------------------------
cpt_mean, cpt_lo, cpt_hi = [], [], []
tphg_mean, tphg_lo, tphg_hi = [], [], []

for d in devices:
    cpt_vals = cpt_df.loc[cpt_df["Device"] == d, "Load Time (s)"].values
    tphg_vals = res_df.loc[res_df["current_device"] == d, "Cached TPHG Load Time (t1)"].values

    m, lo, hi = bootstrap_mean_ci(cpt_vals)
    cpt_mean.append(m); cpt_lo.append(lo); cpt_hi.append(hi)

    m, lo, hi = bootstrap_mean_ci(tphg_vals)
    tphg_mean.append(m); tphg_lo.append(lo); tphg_hi.append(hi)

cpt_mean = np.array(cpt_mean); cpt_lo = np.array(cpt_lo); cpt_hi = np.array(cpt_hi)
tphg_mean = np.array(tphg_mean); tphg_lo = np.array(tphg_lo); tphg_hi = np.array(tphg_hi)

# -------------------------------
# Storage sizes per device
# -------------------------------
sizes_df = collect_sizes(BASE_DIR, allow_devices=set(devices))
sizes_df = sizes_df.sort_values("Device", key=lambda s: s.map(dev_key))
# Align order with load-time devices list (in case of mismatch)
sizes_df = sizes_df.set_index("Device").reindex(devices).dropna().reset_index()

# -------------------------------
# Metrics computation
# -------------------------------
# Speedup per device (based on CI means above)
speedups = cpt_mean / tphg_mean
gm = gmean(speedups)
gm_lo, gm_hi = boot_ci(speedups, fn=gmean)

# Mean storage reduction across devices (only where sizes available)
stor_red = None
stor_red_lo = stor_red_hi = None
mean_tphg_size = mean_cpt_size = np.nan

if len(sizes_df):
    mean_tphg_size = float(sizes_df["TPHG_Size_MB"].mean())
    mean_cpt_size  = float(sizes_df["CPT_Size_MB"].mean())
    stor_red_vec   = 1.0 - (sizes_df["TPHG_Size_MB"].values / sizes_df["CPT_Size_MB"].values)
    stor_red       = float(np.mean(stor_red_vec))
    stor_red_lo, stor_red_hi = boot_ci(stor_red_vec, fn=np.mean)

# -------------------------------
# Plot 1: Load-time (with 95% CI)
# -------------------------------
fig, ax = plt.subplots(figsize=(14, 8))
x = np.arange(len(devices))
bar_width = 0.4
patterns = ["//", "\\"]

cpt_yerr = np.vstack([cpt_mean - cpt_lo, cpt_hi - cpt_mean])
tphg_yerr = np.vstack([tphg_mean - tphg_lo, tphg_hi - tphg_mean])

ax.bar(
    x - bar_width/2, cpt_mean,
    width=bar_width, label="CPT Load Time (s)",
    color="skyblue", edgecolor="black", hatch=patterns[0],
    yerr=cpt_yerr, capsize=6, ecolor="black"
)
ax.bar(
    x + bar_width/2, tphg_mean,
    width=bar_width, label="TPHG Load Time (s)",
    color="lightgreen", edgecolor="black", hatch=patterns[1],
    yerr=tphg_yerr, capsize=6, ecolor="black"
)

ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(devices, rotation=45, ha="right", fontsize=font2['size'], family=font2['family'])
ax.set_xlabel("Device Names", fontsize=font1['size'], family=font1['family'])
ax.set_ylabel("Load Time (s, Log Scale)", fontsize=font1['size'], family=font1['family'])
ax.yaxis.set_tick_params(labelsize=30)
ax.legend(fontsize=font0['size'])
ax.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig("cpt_vs_tphg_load_time_comparison_log.png", dpi=300)
plt.close(fig)

# -------------------------------
# Plot 2: Storage (no per-device CI)
# -------------------------------
# Use available sizes_df; if some devices lacked size files, plot what we have.
if len(sizes_df):
    fig, ax = plt.subplots(figsize=(14, 8))
    x = np.arange(len(sizes_df))
    ax.bar(
        x, sizes_df["TPHG_Size_MB"].values,
        width=0.4, label='Max TPHG Size (MB)',
        align='center', hatch='//', color='skyblue', edgecolor='black'
    )
    ax.bar(
        x + 0.4, sizes_df["CPT_Size_MB"].values,
        width=0.4, label='Conditional Probability Table (MB)',
        align='center', hatch='\\', color='lightgreen', edgecolor='black'
    )
    ax.set_yscale('log')
    ax.set_xlabel('Device Names', fontsize=font1['size'], family=font1['family'])
    ax.set_ylabel('Size (MB, Log Scale)', fontsize=font1['size'], family=font1['family'])
    ax.set_xticks(x + 0.2)
    ax.set_xticklabels(sizes_df["Device"].tolist(), rotation=45, ha='right', fontsize=font2['size'], family=font2['family'])
    ax.legend(fontsize=font0['size'], loc="upper right")
    ax.tick_params(axis='y', labelsize=font2['size'])
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('cptwithtphgstorage.png', dpi=300)
    plt.close(fig)

# -------------------------------
# Save a concise, paper-ready summary
# -------------------------------
lines = []
lines.append("=== Metrics Summary ===")
lines.append(f"Devices included (load-time): {len(devices)} -> {', '.join(devices)}")
if len(sizes_df):
    lines.append(f"Devices with size data: {len(sizes_df)} -> {', '.join(sizes_df['Device'].tolist())}")
lines.append("")
lines.append(f"Geometric-mean speedup (CPT/TPHG): {gm:.3f}x (95% CI {gm_lo:.3f}–{gm_hi:.3f})")
if stor_red is not None:
    lines.append(f"Mean storage reduction: {stor_red*100:.2f}% (95% CI {stor_red_lo*100:.2f}–{stor_red_hi*100:.2f}%)")
    lines.append(f"Mean CPT size:  {mean_cpt_size:.3f} MB")
    lines.append(f"Mean TPHG size: {mean_tphg_size:.3f} MB")
else:
    lines.append("Storage sizes not available for the selected devices; skipping storage metrics.")

with open("metrics_summary.txt","w") as f:
    f.write("\n".join(lines))

print("\n".join(lines))
print("\nSaved figures:")
print(" - cpt_vs_tphg_load_time_comparison_log.png")
if len(sizes_df):
    print(" - cptwithtphgstorage.png")
print("Also wrote: metrics_summary.txt")


## Section 10: Timing Breakdown by Disease — Alternative Visualization

Generate a per-condition breakdown of TPHG load, Viterbi, and fallback times with 95% CI error bars using scipy. Useful for cross-disease timing comparison figures.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats
import textwrap


conditions = [
    "Stroke",
    "Myocardial infarction",
    "Suspected lung cancer",
    "Non-small cell lung cancer",
    "Seizure disorder"
]


disease_name_mapping = {
    "Stroke": "Stroke",
    "Myocardialinfarction(disorder)": "Myocardial infarction",
    "Suspectedlungcancer(situation)": "Suspected lung cancer",
    "Non-smallcelllungcancer(disorder)": "Non-small cell lung cancer",
    "Seizuredisorder": "Seizure disorder",
    "Myocardial infarction (disorder)": "Myocardial infarction",
    "Suspected lung cancer (situation)": "Suspected lung cancer",
    "Non-small cell lung cancer (disorder)": "Non-small cell lung cancer",
    "Seizure disorder": "Seizure disorder"
}


results_df = pd.read_csv("results.csv")
results_df["Disease Name"] = results_df["Disease Name"].map(disease_name_mapping)


metrics = ["Cached TPHG Load Time (t1)", "Backward Viterbi Time (t2)", "Fallback Path Time (t_fallback)"]
custom_legend_names = ["TPHG Load Time", "Backward Viterbi Time", "Fallback Path Time"]


grouped_df = results_df.groupby("Disease Name")[metrics].agg(['mean', 'count', 'std']).reset_index()


for metric in metrics:
    grouped_df[f"{metric} 95% CI"] = stats.t.ppf(0.975, grouped_df[(metric, 'count')] - 1) * grouped_df[(metric, 'std')] / np.sqrt(grouped_df[(metric, 'count')])


grouped_df.columns = [' '.join(col).strip() if col[1] else col[0] for col in grouped_df.columns]


greedy_data = pd.read_csv("greedy_walk_summary_with_resources.csv")
greedy_data["Disease Name"] = greedy_data["condition_name"].map(disease_name_mapping)


merged_df = grouped_df.merge(greedy_data, left_on="Disease Name", right_on="Disease Name", how="inner")


wrapped_labels = [
    "\n".join(textwrap.wrap(disease_name, width=15)) for disease_name in merged_df["Disease Name"]
]


font0 = {'family': 'Times New Roman', 'size': 24}
font1 = {'family': 'Times New Roman', 'size': 28}
font2 = {'family': 'Times New Roman', 'size': 26}

x = np.arange(len(merged_df["Disease Name"]))  
bar_width = 0.4  
colors = ["lightblue", "lightgreen", "salmon"]  
patterns = ["///", "\\\\", "**", "xx"]  


fig, ax = plt.subplots(figsize=(14, 7))


lave_bottom = np.zeros(len(merged_df["Disease Name"]))
for i, (metric, legend_name) in enumerate(zip(metrics, custom_legend_names)):
    ax.bar(
        x,
        merged_df[f"{metric} mean"],
        bottom=lave_bottom,
        width=bar_width,
        label=legend_name,  
        hatch=patterns[i],
        color=colors[i],
        edgecolor="black",
        capsize=5
    )
    lave_bottom += merged_df[f"{metric} mean"]


ax.bar(
    x + bar_width,
    merged_df["total_time"],
    width=bar_width,
    label="Greedy Walk Time (Centralized)",
    hatch=patterns[-1],
    color="lightgray",
    edgecolor="black"
)


ax.set_yscale("log")  
ax.set_ylim(1e-3, None)  
ax.set_xticks(x + bar_width / 2)
ax.set_xticklabels(wrapped_labels, rotation=0, ha='center', **font2)  
ax.yaxis.set_tick_params(labelsize= 26) 
ax.set_xlabel("Disease Name", **font1)
ax.set_ylabel("Log Time (s)", **font1)
ax.legend(
    fontsize=20,  
    loc="upper right",
    ncol=1
)
ax.grid(axis="y", linestyle="--", alpha=0.7)


plt.tight_layout()


plt.savefig("disease_comparison_plot_with_custom_legend.png")
plt.show()


## Section 11: CasFly vs Greedy Baseline Comparison

Load `greedy_walk_summary_with_resources.csv` and `results.csv`. Normalize disease names and align for side-by-side comparison of CasFly chain results against the greedy-walk baseline.

In [ ]:
import pandas as pd
import numpy as np


disease_name_mapping = {
    "Stroke": "Stroke",
    "Myocardialinfarction(disorder)": "Myocardial infarction (disorder)",
    "Suspectedlungcancer(situation)": "Suspected lung cancer (situation)",
    "Non-smallcelllungcancer(disorder)": "Non-small cell lung cancer (disorder)",
    "Seizuredisorder": "Seizure disorder",
    "Myocardial infarction (disorder)": "Myocardial infarction (disorder)",
    "Suspected lung cancer (situation)": "Suspected lung cancer (situation)",
    "Non-small cell lung cancer (disorder)": "Non-small cell lung cancer (disorder)",
    "Seizure disorder": "Seizure disorder"
}


results_df = pd.read_csv("results.csv")
greedy_data = pd.read_csv("greedy_walk_summary_with_resources.csv")


results_df["Disease Name"] = results_df["Disease Name"].map(disease_name_mapping)
greedy_data["Disease Name"] = greedy_data["condition_name"].map(disease_name_mapping)


metrics = ["Cached TPHG Load Time (t1)", "Backward Viterbi Time (t2)", "Fallback Path Time (t_fallback)"]


grouped_df = results_df.groupby("Disease Name")[metrics].mean().reset_index()


merged_df = grouped_df.merge(greedy_data, on="Disease Name", how="inner")


merged_df["LaVE Total Time"] = merged_df["Cached TPHG Load Time (t1)"] + \
                               merged_df["Backward Viterbi Time (t2)"] + \
                               merged_df["Fallback Path Time (t_fallback)"]


merged_df["Percentage Improvement"] = ((merged_df["total_time"] - merged_df["LaVE Total Time"]) / 
                                       merged_df["total_time"]) * 100


print(merged_df[["Disease Name", "total_time", "LaVE Total Time", "Percentage Improvement"]])


## Section 12: CPT vs TPHG Load Time Bar Plot

Final paper figure: grouped bar chart comparing mean CPT load time vs TPHG load time per device, using data from `device_metrics_analysiscptloadtime.csv` and `results.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


font0 = {'family': 'Times New Roman', 'size': 28}
font1 = {'family': 'Times New Roman', 'size': 32}
font2 = {'family': 'Times New Roman', 'size': 30}
font3 = {'family': 'Times New Roman', 'size': 33}


cpt_df = pd.read_csv("device_metrics_analysiscptloadtime.csv")


results_df = pd.read_csv("results.csv")


tphg_load_time = results_df.groupby("current_device")["Cached TPHG Load Time (t1)"].mean()


cpt_load_time = cpt_df.groupby("Device")["Load Time (s)"].mean()


common_devices = cpt_load_time.index.intersection(tphg_load_time.index)
cpt_load_time = cpt_load_time[common_devices]
tphg_load_time = tphg_load_time[common_devices]


merged_load_times = pd.DataFrame({
    "CPT Load Time (s)": cpt_load_time,
    "TPHG Load Time (s)": tphg_load_time
}).dropna()


def sort_device_key(device):
    
    return int(''.join(filter(str.isdigit, device)))

merged_load_times = merged_load_times.reindex(sorted(merged_load_times.index, key=sort_device_key))


fig, ax = plt.subplots(figsize=(14, 8))
x = np.arange(len(merged_load_times))  
bar_width = 0.4
patterns = ["//", "\\"]  


ax.bar(
    x - bar_width / 2,
    merged_load_times["CPT Load Time (s)"],
    width=bar_width,
    label="CPT Load Time (s)",
    color="skyblue",
    edgecolor="black",
    hatch=patterns[0]
)

ax.bar(
    x + bar_width / 2,
    merged_load_times["TPHG Load Time (s)"],
    width=bar_width,
    label="TPHG Load Time (s)",
    color="lightgreen",
    edgecolor="black",
    hatch=patterns[1]
)


ax.set_yscale("log")


ax.set_xticks(x)
ax.set_xticklabels(merged_load_times.index, rotation=45, ha="right", fontsize=font2['size'], family=font2['family'])
ax.set_xlabel("Device Names", fontsize=font1['size'], family=font1['family'])
ax.set_ylabel("Load Time (s, Log Scale)", fontsize=font1['size'], family=font1['family'])
ax.yaxis.set_tick_params(labelsize= 30) 
ax.legend(fontsize=font0['size'])
ax.grid(axis="y", linestyle="--", alpha=0.7)


plt.tight_layout()


plt.savefig("cpt_vs_tphg_load_time_comparison_log.png")
plt.show()
